In [13]:
# All packages I'll need

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

In [14]:
Fd = pd.read_csv("creditcard.csv")

In [15]:
#Normalization

scaler = StandardScaler()
Fd[['Amount_scaled']] = scaler.fit_transform(Fd[['Amount']])
Fd[['Time_scaled']] = scaler.fit_transform(Fd[['Time']])
Fd.drop(['Amount','Time'], axis=1, inplace=True)

In [16]:
#Preparing the data for the model

X = Fd.drop('Class', axis=1)
Y = Fd['Class']

In [17]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state = 42, stratify=Y) 

In [18]:
scale = (Y_train ==0).sum()/(Y_train ==1).sum()

In [19]:
#Trainning the model

My_Fd_Model = xgb.XGBClassifier(
    scale_pos_weight = scale,
    n_estimators = 100,
    max_depth = 6,
    learning_rate = 0.1,
    random_state = 42,
    eval_metrics = "aucpr"
)

In [20]:
My_Fd_Model.fit(X_train,Y_train)

C:\Users\DELL\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:34:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "eval_metrics" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, eval_metrics='aucpr',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None, ...)

In [21]:
Y_pred = My_Fd_Model.predict(X_test)
My_model_prob = My_Fd_Model.predict_proba(X_test)[:,1]

In [22]:
#Model evaluation

print(classification_report(Y_test, Y_pred, target_names=["Normal","Fraud"]))

              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.76      0.84      0.80        98

    accuracy                           1.00     56962
   macro avg       0.88      0.92      0.90     56962
weighted avg       1.00      1.00      1.00     56962



In [23]:
print(roc_auc_score(Y_test, Y_pred))

0.9181387312944311


In [24]:
#Track the experiment with mlflow

mlflow.set_tracking_uri("file:///C:/Users/DELL/Desktop/IA/ML/Projects/Fraud detection/mlruns")
mlflow.set_experiment("My_Fraud_detection")
mlflow.sklearn.autolog()

with mlflow.start_run():
    My_Fd_Model.fit(X_train, Y_train)

    print("Run registed!")

C:\Users\DELL\anaconda3\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
Traceback (most recent call last):
  File "C:\Users\DELL\anaconda3\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DELL\anaconda3\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\D

Run registed!


In [30]:
#Fraud transac test

fraud_test = Fd[Fd["Class"] == 1].iloc[0]
print(fraud_test)

V1              -2.312227
V2               1.951992
V3              -1.609851
V4               3.997906
V5              -0.522188
V6              -1.426545
V7              -2.537387
V8               1.391657
V9              -2.770089
V10             -2.772272
V11              3.202033
V12             -2.899907
V13             -0.595222
V14             -4.289254
V15              0.389724
V16             -1.140747
V17             -2.830056
V18             -0.016822
V19              0.416956
V20              0.126911
V21              0.517232
V22             -0.035049
V23             -0.465211
V24              0.320198
V25              0.044519
V26              0.177840
V27              0.261145
V28             -0.143276
Class            1.000000
Amount_scaled   -0.353229
Time_scaled     -1.988034
Name: 541, dtype: float64


In [29]:
#Normal transac test

normal_test = Fd[Fd["Class"] == 0].iloc[0]
print(normal_test)

V1              -1.359807
V2              -0.072781
V3               2.536347
V4               1.378155
V5              -0.338321
V6               0.462388
V7               0.239599
V8               0.098698
V9               0.363787
V10              0.090794
V11             -0.551600
V12             -0.617801
V13             -0.991390
V14             -0.311169
V15              1.468177
V16             -0.470401
V17              0.207971
V18              0.025791
V19              0.403993
V20              0.251412
V21             -0.018307
V22              0.277838
V23             -0.110474
V24              0.066928
V25              0.128539
V26             -0.189115
V27              0.133558
V28             -0.021053
Class            0.000000
Amount_scaled    0.244964
Time_scaled     -1.996583
Name: 0, dtype: float64
